In [1]:
!pip install sentence-transformers scikit-learn pandas numpy safetensors -q


In [1]:
import pandas as pd

# Pulling the newly cleaned synthetic dataset directly from the repo
df = pd.read_csv("https://raw.githubusercontent.com/kyahikaru/hinglish-prompt-injection-detector/main/training/dataset.csv")

print("--- DATASET INTEGRITY CHECK ---")
print("Total:", len(df))
print(df['label'].value_counts())

--- DATASET INTEGRITY CHECK ---
Total: 2106
label
0    1227
1     879
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

# 70% Train, 30% Temp
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
# Split Temp evenly into 15% Val, 15% Test
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print("--- SPLIT VERIFICATION ---")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

--- SPLIT VERIFICATION ---
Train: 1474
Val: 316
Test: 316


In [3]:
from sentence_transformers import SentenceTransformer

# Load the alpha embedding model
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("Encoding Training Data...")
X_train = model.encode(train_df['text'].tolist(), show_progress_bar=True)

print("Encoding Validation Data...")
X_val = model.encode(val_df['text'].tolist(), show_progress_bar=True)

print("Encoding Test Data...")
X_test = model.encode(test_df['text'].tolist(), show_progress_bar=True)

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Encoding Validation Data...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Encoding Test Data...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred_val = clf.predict(X_val)

print("--- VALIDATION METRICS ---")
print(classification_report(y_val, y_pred_val))

--- VALIDATION METRICS ---
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       184
           1       0.96      0.97      0.96       132

    accuracy                           0.97       316
   macro avg       0.97      0.97      0.97       316
weighted avg       0.97      0.97      0.97       316



In [5]:
from sklearn.metrics import confusion_matrix

y_pred_test = clf.predict(X_test)

print("--- FINAL TEST METRICS ---")
print(classification_report(y_test, y_pred_test))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

--- FINAL TEST METRICS ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       184
           1       0.98      0.98      0.98       132

    accuracy                           0.99       316
   macro avg       0.99      0.99      0.99       316
weighted avg       0.99      0.99      0.99       316

Confusion Matrix:
[[182   2]
 [  2 130]]


In [7]:
!pip install onnx skl2onnx -q
import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input shape matching our embeddings (384 dimensions for MiniLM)
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]

# Convert and serialize
onx = convert_sklearn(clf, initial_types=initial_type)

with open("classifier.onnx", "wb") as f:
    f.write(onx.SerializeToString())

print("Model successfully compiled and saved as classifier.onnx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 31.4 MB/s eta 0:00:00
Model successfully compiled and saved as classifier.onnx
